In [ ]:
from graphs import generate_grid_graph, grid_graph_max_cut, grid_renderer, state_history_renderer, render_state, render_state_prog_gif
from cim import CIM
from solvers import SG3
from graphs import read_graph_from_rudy, eval_max_cut
from schedules import schedule_linear
from transfers import Clipped
import torch
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
torch.manual_seed(0xDEADBEEF + 0)

size = 20
J = generate_grid_graph(size)
max_cut = grid_graph_max_cut(size)
print(f"{max_cut=}")
pump_schedule=schedule_linear(slope=0.01, t_mid=50, max_val = 1.01)
noise_magnitude = 0.003
steps=25000
step_size=0.02

In [ ]:
# foo = CIM(-0.1 * J, solver=SG3)
# foo.solve()
# print(foo.model.state)
# print("SG3:", eval_max_cut(foo.model.state.tolist(), J))

## DOPO CIM

In [ ]:
dopo_cim = CIM(-0.1 * J, pump_schedule=pump_schedule)
dopo_cim.solve(noise_magnitude=noise_magnitude, steps=steps, step_size=step_size)
while (cut := eval_max_cut(dopo_cim.model.state.tolist(), J)) != max_cut:
    print(cut)
    dopo_cim.solve(noise_magnitude=noise_magnitude, steps=steps, step_size=step_size)
print(dopo_cim.model.state)
print("DOPO:", eval_max_cut(dopo_cim.model.state.tolist(), J))

In [ ]:
fig, ax = plt.subplots()
ax.plot(dopo_cim.model.result.state_history)
ax.set_xlabel("Steps")
ax.set_ylabel("CIM Spin (a.u.)")
fig.savefig('ims/DOPO_CIM_Sqare_Lattice_States.png', dpi=900)

In [ ]:
for step in (100, 1500, 2000, 5000, 24990):
    fig, _ = render_state(step, dopo_cim.model.result.state_history, [grid_renderer], ncols=1, with_domain_border=True)
    fig.savefig(f"ims/DOPO_CIM_Sqare_Lattice_Grid_{size}_step_{step}.png", dpi=900, transparent=True)

In [ ]:
render_state_prog_gif(dopo_cim.model.result.state_history, [grid_renderer, state_history_renderer], 
                      with_domain_border=True, ncols=2, step=500, num_rendered=52,
                      filename='ims/DOPO_CIM_Sqare_Lattice_All.gif')

In [ ]:
render_state_prog_gif(dopo_cim.model.result.state_history, [grid_renderer], 
                      title="CIM Square Lattice Progression", ncols=1, step=250,
                      with_domain_border=True,
                      filename='ims/DOPO_CIM_Sqare_Lattice_Grid.gif')

## Clipped CIM

In [ ]:
torch.manual_seed(0xDEADBEEF + 0)
foo = CIM(-0.1 * J, pump_schedule=pump_schedule, transfer=Clipped)
foo.solve(noise_magnitude=noise_magnitude, steps=steps, step_size=step_size)
# while (cut := eval_max_cut(foo.model.state.tolist(), J)) != max_cut:
#     print(cut)
#     foo.solve(noise_magnitude=noise_magnitude, steps=steps, step_size=step_size)
print(foo.model.state)
print("DOPO:", eval_max_cut(foo.model.state.tolist(), J))

In [ ]:
fig, ax = plt.subplots()
ax.plot(foo.model.result.state_history)
ax.set_xlabel("Steps")
ax.set_ylabel("CIM Spin (a.u.)")
fig.savefig('ims/Clipped_CIM_Sqare_Lattice_States.png', dpi=900)

In [ ]:
for step in (100, 1500, 2000, 5000, 24990):
    fig, _ = render_state(step, foo.model.result.state_history, [grid_renderer], ncols=1, with_domain_border=True)
    fig.savefig(f"ims/Clipped_CIM_Sqare_Lattice_Grid_{size}_step_{step}.png", dpi=900, transparent=True)

In [ ]:
render_state_prog_gif(foo.model.result.state_history, [grid_renderer, state_history_renderer], 
                      with_domain_border=True, ncols=2, step=500, num_rendered=52,
                      filename='ims/Clipped_CIM_Sqare_Lattice_All.gif')

In [ ]:
render_state_prog_gif(foo.model.result.state_history, [grid_renderer], 
                      title="CIM Square Lattice Progression", ncols=1, step=1000,
                      with_domain_border=True, filename="ims/Clipped_CIM_Sqare_Lattice_Grid.gif")